# LangChain 实战：起名大师

本 Notebook 演示 4 个核心功能模块：
1. **创建 LLM**：调用智谱 GLM API 给孩子起具有**中国特色**的名字
2. **自定义提示词模板**：支持参数传入，起具有**美国特色**的名字
3. **输出解释器**：将 LLM 输出格式化为**数组/JSON**等结构化数据
4. **交互式实例**：以**人机对话**形式，根据用户输入的性别/数量输出名字数组

## 前置依赖安装
### 1.创建 venv 虚拟环境	✅
### 2.安装 langchain、langchain-openai、pydantic	✅
### 3.安装 ipykernel	✅
### 4.注册 Jupyter 内核 Python (ai-agent-env)	✅

```bash

1.
python -m venv d:\APP\npSpace\ai-agent\venv

2.
d:\APP\npSpace\ai-agent\venv\Scripts\python.exe -m pip install langchain langchain-openai pydantic -i https://pypi.tuna.tsinghua.edu.cn/simple

3.
d:\APP\npSpace\ai-agent\venv\Scripts\python.exe -m pip install ipykernel -i https://pypi.tuna.tsinghua.edu.cn/simple

4.
d:\APP\npSpace\ai-agent\venv\Scripts\python.exe -m ipykernel install --user --name=ai-agent-env --display-name "Python (ai-agent-env)"
```

## API 配置
- **API Key**: `729fed59a61a444bb1362471dfbb547d.ZaHfcmzo7Ukhs2f6`
- **模型**: `GLM-4.5-Air` / `GLM-4.7-Flash`
- **Base URL**: `https://open.bigmodel.cn/api/paas/v4/`

In [5]:
# ==========================================
# 基础导入与 LLM 初始化
# ==========================================
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# GLM API 配置（智谱 OpenAI 兼容接口）
GLM_API_KEY = "729fed59a61a444bb1362471dfbb547d.ZaHfcmzo7Ukhs2f6"
GLM_MODEL = "GLM-4.5-Air"      # 可切换为 "GLM-4.7-Flash"
GLM_BASE_URL = "https://open.bigmodel.cn/api/paas/v4/"

# 初始化大模型
llm = ChatOpenAI(
    model=GLM_MODEL,
    api_key=GLM_API_KEY,
    base_url=GLM_BASE_URL,
    temperature=0.8
)

print(f"✅ LLM 初始化完成 | 模型: {GLM_MODEL}")

ModuleNotFoundError: No module named 'langchain_openai'

## 1. 创建 LLM —— 给孩子起具有中国特色的名字

直接调用大模型，通过 `SystemMessage` 设定角色，通过 `HumanMessage` 发送指令。

In [2]:
# 1. 创建 LLM —— 给孩子起具有中国特色的名字
messages = [
    SystemMessage(content="你是一个起名大师，精通中国传统文化，擅长起具有中国特色的名字。"),
    HumanMessage(content="请给我 3 个具有中国特色的男孩名字，并简单解释每个名字的寓意。")
]

response = llm.invoke(messages)
print(response.content)

NameError: name 'SystemMessage' is not defined

## 2. 自定义提示词模板 —— 给孩子起具有美国特色的名字

使用 `PromptTemplate` 将提问上下文**模板化**，支持通过参数传入：
- `{county}`: 国家/地区风格
- `{gender}`: 男孩 / 女孩
- `{boy}`: 男孩示例名
- `{girl}`: 女孩示例名
- `{count}`: 名字数量

In [ ]:
from langchain_core.prompts import PromptTemplate

# 2. 自定义提示词模板 —— 起具有美国特色的名字
prompt_template = PromptTemplate.from_template(
    "你是一个起名大师，请模仿示例起 {count} 个具有 {country} 特色的 {gender} 名字，\n"
    "示例：男孩经常被叫做 {boy}，女孩经常被叫做 {girl}。\n"
    "请列出每个名字，并简单说明其寓意。"
)

# 通过参数传入模板变量
prompt = prompt_template.format(
    count="3",
    country="美国",
    gender="男孩",
    boy="Jack, Michael",
    girl="Emily, Sophia"
)

print("========== 生成的提示词 ==========")
print(prompt)
print("\n========== LLM 输出 ==========")

response = llm.invoke([HumanMessage(content=prompt)])
print(response.content)

## 3. 输出解释器 —— 将 LLM 输出格式化为数组

让 LLM 给孩子起 **4 个有中国特色的名字**，并以**数组格式**输出而不是纯文本。
这里使用 `JsonOutputParser` + `Pydantic` 模型实现结构化数据输出。

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

# 定义输出数据结构
class NameItem(BaseModel):
    name: str = Field(description="名字")
    meaning: str = Field(description="名字的寓意")
    gender: str = Field(description="性别：男 / 女")

class NameList(BaseModel):
    names: List[NameItem] = Field(description="名字列表")

parser = JsonOutputParser(pydantic_object=NameList)

# 构建带格式说明的提示词模板
prompt_template = PromptTemplate.from_template(
    "你是一个起名大师，请给我 {count} 个具有中国特色的 {gender} 名字。\n"
    "请以 JSON 格式输出，严格遵循以下格式：\n"
    "{format_instructions}\n"
    "要求：\n"
    "- name: 名字\n"
    "- meaning: 寓意（30字以内）\n"
    "- gender: 性别（男/女）\n"
    "- 只输出 JSON，不要输出其他内容。",
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

prompt = prompt_template.format(count="4", gender="女孩")
print("========== 提示词 ==========")
print(prompt)

response = llm.invoke([HumanMessage(content=prompt)])
print("\n========== LLM 原始输出 ==========")
print(response.content)

print("\n========== 解析后的数组（Python 对象）==========")
result = parser.parse(response.content)
print(result)

# 提取纯名字数组
name_array = [item["name"] for item in result["names"]]
print("\n========== 纯名字数组 ==========")
print(name_array)
print(f"类型: {type(name_array)}")

## 4. 交互式实例 —— 人机对话形式输出名字数组

封装一个函数 `generate_names()`，实现：
- 根据用户输入的 **性别** 输出对应名字
- 根据用户输入的 **数量** 输出对应个数
- 输出格式为**数组**（JSON 结构化数据）
- 支持切换风格（中国特色 / 美国特色等）

In [ ]:
# 4. 交互式实例 —— 人机对话形式
def generate_names(gender: str = "男孩", count: int = 4, style: str = "中国特色") -> dict:
    """
    起名助手：根据用户输入生成名字数组
    
    :param gender: 性别，如 "男孩" / "女孩"
    :param count: 名字数量
    :param style: 名字风格，如 "中国特色" / "美国特色"
    :return: 结构化名字数据（dict）
    """
    parser = JsonOutputParser(pydantic_object=NameList)
    
    prompt_template = PromptTemplate.from_template(
        "你是一个友好的起名助手，正在与用户对话。\n"
        "用户希望给 {gender} 孩子起 {count} 个具有 {style} 的名字。\n"
        "请以 JSON 格式输出，严格遵循以下格式：\n"
        "{format_instructions}\n"
        "注意：只输出 JSON，不要返回其他内容。",
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    
    prompt = prompt_template.format(
        gender=gender,
        count=str(count),
        style=style
    )
    
    response = llm.invoke([HumanMessage(content=prompt)])
    result = parser.parse(response.content)
    return result


# ========== 示例调用 ==========
print("\n【示例 1】起 4 个中国特色男孩名字")
print("-" * 50)
result1 = generate_names(gender="男孩", count=4, style="中国特色")
names1 = [item["name"] for item in result1["names"]]
print(f"名字数组: {names1}")
print(f"完整数据: {result1}")

print("\n【示例 2】起 4 个美国特色女孩名字")
print("-" * 50)
result2 = generate_names(gender="女孩", count=4, style="美国特色")
names2 = [item["name"] for item in result2["names"]]
print(f"名字数组: {names2}")
print(f"完整数据: {result2}")

print("\n【示例 3】起 3 个中国特色女孩名字")
print("-" * 50)
result3 = generate_names(gender="女孩", count=3, style="中国特色")
names3 = [item["name"] for item in result3["names"]]
print(f"名字数组: {names3}")
print(f"完整数据: {result3}")

---

## 附：模型切换说明

如需切换模型，只需修改初始化代码中的 `GLM_MODEL` 变量：

```python
GLM_MODEL = "GLM-4.7-Flash"   # 更快更轻量
# GLM_MODEL = "GLM-4.5-Air"   # 默认推荐
```